# 17 — NLP & AI Applications

## 📓 Interactive Notebook · Module 13 · AI/LLM

In this notebook, you'll learn:
1. **NLP workflow:** Text → Preprocessing → Features → Model → Prediction
2. **Text preprocessing** (cleaning, tokenization)
3. **Feature extraction** with TF-IDF
4. **Text classification** with scikit-learn
5. **Sentiment analysis** application
6. **Building NLP Streamlit apps**

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Understand the NLP deployment pipeline
- Preprocess text for machine learning
- Build text classifiers with TF-IDF
- Create sentiment analysis apps
- Handle text input validation

## 📋 Prerequisites

- Modules 01–12 completed
- Basic ML knowledge (classification, train/test split)
- Python string operations

---

## 💡 The NLP Workflow

```
1. Text Input → 2. Preprocessing → 3. Feature Extraction → 4. Model → 5. Prediction
   (User)         (Clean/Token)      (TF-IDF)           (Trained)    (Display)
```

---

## 💡 Step 1: Text Preprocessing

Clean and normalize text before feature extraction.

In [ ]:
import re
import streamlit as st

st.header("📝 Text Preprocessing")

def clean_text(text):
    """Basic text cleaning."""
    # Lowercase
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove special characters (keep letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Demo
sample_texts = [
    "This is GREAT!!! Visit https://example.com for more.",
    "<p>Hello World</p> This is a <b>test</b>.",
    "LOVE this product!!! Best purchase ever!!!"
]

for text in sample_texts:
    cleaned = clean_text(text)
    st.write(f"**Original:** {text}")
    st.write(f"**Cleaned:** {cleaned}")
    st.write("---")

---

## 💡 Step 2: Feature Extraction with TF-IDF

Convert text to numerical features.

In [ ]:
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

st.header("📊 TF-IDF Feature Extraction")

# Sample documents
documents = [
    "I love this product, it is amazing",
    "This is terrible, worst experience ever",
    "Great quality, highly recommend",
    "Awful customer service, never again",
    "Fantastic value for money"
]

labels = ["positive", "negative", "positive", "negative", "positive"]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=10, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(documents)

# Show feature names
feature_names = vectorizer.get_feature_names_out()
st.write("**Top Features:**", list(feature_names))

# Show TF-IDF scores
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"Doc {i+1}" for i in range(len(documents))]
)
st.dataframe(tfidf_df.round(3))

st.info("💡 TF-IDF scores show word importance: high = frequent in doc, rare overall")

---

## 💡 Step 3: Train a Text Classifier

Build a complete text classification pipeline.

In [ ]:
import streamlit as st
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

st.header("🎯 Train Text Classifier")

# Sample training data (in practice, use a real dataset)
train_texts = [
    # Positive
    "I love this product, it is amazing",
    "Great quality, highly recommend",
    "Fantastic value for money",
    "Best purchase I ever made",
    "Excellent service and fast delivery",
    # Negative
    "This is terrible, worst experience",
    "Awful customer service, never again",
    "Complete waste of money",
    "Product broke after one day",
    "Very disappointed with quality"
]

train_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]  # 1=positive, 0=negative

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42
)

# Create pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train
pipeline.fit(X_train, y_train)

# Evaluate
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

st.write(f"**Accuracy:** {accuracy:.2%}")

# Save model
joblib.dump(pipeline, "text_classifier.joblib")
st.success("✅ Model trained and saved!")

---

## 💡 Step 4: Make Predictions

Load model and predict on new text.

In [ ]:
import streamlit as st
import joblib
import re

st.header("🔮 Text Prediction")

@st.cache_resource
def load_model():
    """Load trained pipeline."""
    return joblib.load("text_classifier.joblib")

def clean_text(text):
    """Clean text (must match training)."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load model
pipeline = load_model()

# Input
text = st.text_area(
    "Enter text to classify",
    value="This product is amazing!",
    height=100
)

if st.button("Classify"):
    # Validate
    if not text.strip():
        st.error("Please enter some text")
    else:
        # Preprocess (must match training!)
        cleaned = clean_text(text)
        
        # Predict
        prediction = pipeline.predict([cleaned])[0]
        probabilities = pipeline.predict_proba([cleaned])[0]
        
        # Display
        if prediction == 1:
            st.success(f"😊 Positive ({probabilities[1]:.1%} confidence)")
        else:
            st.error(f"😞 Negative ({probabilities[0]:.1%} confidence)")
        
        # Confidence breakdown
        st.write("**Confidence:**")
        st.progress(probabilities[1], text=f"Positive: {probabilities[1]:.1%}")
        st.progress(probabilities[0], text=f"Negative: {probabilities[0]:.1%}")

---

## ⚠️ Preprocessing Consistency

**Critical:** The preprocessing at inference MUST match training.

In [ ]:
import streamlit as st

st.header("⚠️ Preprocessing Consistency")

st.subheader("❌ WRONG: Different preprocessing")
st.code('''
# Training: lowercase + remove special chars
def preprocess_train(text):
    return text.lower().strip()

# Inference: different preprocessing!
def preprocess_inference(text):
    return text.strip()  # Missing lowercase!
# Problem: model expects lowercase input
''', language="python")

st.subheader("✅ CORRECT: Same preprocessing")
st.code('''
# Define once, use everywhere
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\\s]', '', text)
    return text

# Training
X_train = [clean_text(t) for t in train_texts]

# Inference
cleaned = clean_text(user_input)  # Same function!
prediction = model.predict([cleaned])
''', language="python")

st.warning("⚠️ Save your preprocessing function or include it in the pipeline!")

---

## 💡 Input Validation

Validate text input before prediction.

In [ ]:
import streamlit as st

st.header("💡 Input Validation")

def validate_text_input(text, max_length=5000):
    """Validate text input."""
    errors = []
    
    if not text or not text.strip():
        errors.append("Text cannot be empty")
    
    if len(text) > max_length:
        errors.append(f"Text too long (max {max_length} characters)")
    
    # Check for suspicious patterns
    suspicious = ['<script>', 'javascript:', 'onerror=']
    for pattern in suspicious:
        if pattern.lower() in text.lower():
            errors.append("Text contains suspicious content")
    
    return errors

# Demo
text = st.text_area("Test input validation", "Enter text here...")

if st.button("Validate"):
    errors = validate_text_input(text)
    if errors:
        for error in errors:
            st.error(f"❌ {error}")
    else:
        st.success("✅ Input is valid")

---

## 💡 Batch Processing

Process multiple documents from uploaded file.

In [ ]:
import streamlit as st
import pandas as pd
import joblib
import re

st.header("📄 Batch Document Processing")

# Load model
pipeline = joblib.load("text_classifier.joblib")

# Sample data for download
sample = pd.DataFrame({
    "text": [
        "This product is amazing!",
        "Terrible experience, very disappointed",
        "Highly recommend to everyone",
        "Worst purchase ever made"
    ]
})
csv = sample.to_csv(index=False)
st.download_button("📥 Download Sample", csv, "sample_texts.csv", "text/csv")

# Upload
uploaded = st.file_uploader("Upload CSV with text column", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)
    st.write("**Preview:**")
    st.dataframe(df.head())
    
    # Select text column
    text_col = st.selectbox("Select text column", df.columns)
    
    if st.button("Process All"):
        # Clean and predict
        cleaned_texts = [clean_text(str(t)) for t in df[text_col]]
        predictions = pipeline.predict(cleaned_texts)
        probabilities = pipeline.predict_proba(cleaned_texts)
        
        # Add results
        df["prediction"] = ["Positive" if p == 1 else "Negative" for p in predictions]
        df["confidence"] = probabilities.max(axis=1)
        
        st.success(f"✅ Processed {len(df)} documents")
        st.dataframe(df)
        
        # Download
        csv = df.to_csv(index=False)
        st.download_button("📥 Download Results", csv, "results.csv", "text/csv")

---

## 🎯 Model Interpretation

Show which words drive predictions.

In [ ]:
import streamlit as st
import pandas as pd

st.header("📊 Model Interpretation")

def show_top_features(pipeline, class_names, top_n=10):
    """Display top features for each class."""
    feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
    coefficients = pipeline.named_steps['classifier'].coef_[0]
    
    # Top positive features
    st.subheader("Top Positive Features")
    top_pos = coefficients.argsort()[-top_n:][::-1]
    pos_df = pd.DataFrame({
        "Feature": [feature_names[i] for i in top_pos],
        "Score": [coefficients[i] for i in top_pos]
    })
    st.bar_chart(pos_df.set_index("Feature")["Score"])
    
    # Top negative features
    st.subheader("Top Negative Features")
    top_neg = coefficients.argsort()[:top_n]
    neg_df = pd.DataFrame({
        "Feature": [feature_names[i] for i in top_neg],
        "Score": [coefficients[i] for i in top_neg]
    })
    st.bar_chart(neg_df.set_index("Feature")["Score"])

# Show features
show_top_features(pipeline, ["Negative", "Positive"])

---

## ⚠️ Common Mistakes

### Mistake 1: Preprocessing Mismatch
```python
# ❌ WRONG
# Training: text.lower()
# Inference: text  # Not lowercased!

# ✅ CORRECT
# Use the same function everywhere
```

### Mistake 2: Fitting Vectorizer at Inference
```python
# ❌ WRONG
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform([user_input])  # fit_transform!

# ✅ CORRECT
X = vectorizer.transform([user_input])  # transform only!
```

### Mistake 3: Not Handling Edge Cases
```python
# ❌ WRONG
prediction = model.predict([user_input])[0]

# ✅ CORRECT
if user_input.strip():
    cleaned = clean_text(user_input)
    prediction = model.predict([cleaned])[0]
else:
    st.error("Please enter some text")
```

---

## 🎯 Challenges

### Challenge 1: Multi-class Classifier
Extend the classifier to handle 3+ categories (positive, negative, neutral).

### Challenge 2: Word Cloud Visualization
Add word cloud visualization for input text.

### Challenge 3: Model Comparison
Compare TF-IDF vs. other vectorization methods.

In [ ]:
# Challenge 1: Multi-class Classifier
import streamlit as st

st.write("TODO: Build a multi-class text classifier")

# Your code here


---

## 📝 Key Takeaways

1. **NLP workflow:** Text → Preprocess → Vectorize → Predict → Display

2. **TF-IDF** converts text to numerical features based on word importance

3. **Preprocessing must match** — use the same cleaning function for training and inference

4. **Use sklearn Pipeline** — bundle vectorizer and model together

5. **Validate inputs** — check for empty text, length limits, suspicious content

6. **Show confidence** — probabilities help users understand predictions

7. **Same ML patterns apply** — train offline, cache models, handle errors

---

## 📚 Further Reading

- [Scikit-learn Text Features](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [NLTK Documentation](https://www.nltk.org/)

---

## 🔗 Related Materials

- 📖 Reading: [17 — NLP & AI Applications](../readings/17_nlp_ai_applications.md)
- ✏️ Exercise: [17 — NLP Workshop](../exercises/17_nlp_workshop.py)
- 🖥️ Demo App: [17 — Sentiment Analyzer](../apps/17_sentiment_app.py)
- 📝 Quiz: [13 — NLP & AI](../quizzes/13_nlp_ai.md)
- 🚀 Project: [P07 — RAG Document Chat](../projects/P07_rag_document_chat.md)